# Neural network for Collaborative Filtering (NCF), 2017


models
- gmf
- mlp: 32 -> 16 -> 8
- neumf: gmf + mlp

exp setting
- activation: relu
- loss: log loss
- mlp init: gaussian(0, 0.01), adam
- neulml init: mlp, gmf pre-trained, sgd
- metrics: HR@10, NDCG@10
- model selection: leave-one-out
- negative sampling 

In [3]:
import os, json, bson

import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf

2023-07-18 14:32:43.320671: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-18 14:32:43.446663: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-18 14:32:43.451142: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/usr/local/cuda-11.7/lib64:/usr/local/cuda/extras/CUPTI/:/usr/local/cuda-11.7/lib6

# Data Load 

In [4]:
pinterest_dir = '../data/pinterest_iccv/'

In [5]:
f = open(os.path.join(pinterest_dir, 'subset_iccv_board_pins.bson'), 'rb')
board_pins = []
for rows in bson.decode_file_iter(f):
    board_pins.append(rows)
len(board_pins)

46000

In [6]:
# user = board_pins[0]['board_id']
# pinned = board_pins[0]['pins']

# Prepare Train/Test data

- leave-one-out evaluation (positive)
    - 테스트 데이터: 유저의 latest 데이터
    - 학습 데이터: 유저의 나머지 데이터
- 학습 시에는 모든 negative 데이터를 포함
- 평가 시에는 positive test data 하나와 나머지 샘플링한 unobserved 100개 데이터를 합하여 랭킹
- 위 랭킹 결과의 top k에 대해서 positive 가 포함되어 있으면 hit, 그 위치가 어디인지를 NDCG가 측정

In [7]:
all_pins = set()

for b in board_pins:
    for p in b['pins']:
        all_pins.add(p)

all_pins = list(all_pins)
len(all_pins)

2565241

In [8]:
pins = [len(b['pins']) for b in board_pins]
num_negative = int(sum(pins)/len(pins))
num_negative

(2519242, 46000, 2519242, 46000)

In [9]:
train = [] # user, item, score
test = [] # user, item, score

for b in tqdm(board_pins):
    train.extend([[b['board_id'], p, 1] for p in b['pins'][:-1]])
    test.append([b['board_id'], b['pins'][-1], 1])

(46000, 2565241)

In [ ]:
def sampling(num_sample, all_pins, without, user_board):
    idx = 0
    sampled = []
    while idx < num_sample:
        randint = np.random.randint(0, 5043)
        if all_pins[randint] in without:
            continue
        else:
            sampled.append([user_board, all_pins[randint], 0])
            idx += 1
    return sampled

In [10]:
for b in tqdm(board_pins):
    train.extend(sampling(num_negative, all_pins, b['pins'], b['board_id']))
    test.extend(sampling(100, all_pins, b['pins'], b['board_id']))

In [ ]:
train, test = np.array(train), np.array(test)
train.shape, test.shape

샘플링은 유저별로 55번씩하면 되는데 positive에 있는 데이터가 들어가면 안됨

In [ ]:
all_users = {tp[0] for tp in np.concatenate([train,test], axis=0)}
all_items = {tp[1] for tp in np.concatenate([train,test], axis=0)}
len(all_users), len(all_items)

# Models

In [13]:
class NCF(tf.keras.Model):

    def __init__(self):
        pass

In [14]:
# gmf
class GMF(NCF):
    def __call__(self):
        pass

# gmf
class MLP(NCF):
    def __call__(self):
        pass

# gmf
class NeuMF(NCF):
    def __call__(self):
        pass

In [15]:
def get_gmf(user_input_size, item_input_size, embedding_size):
    
    user_input = tf.keras.layers.Input(shape=user_input_size, dtype="float32")
    item_input = tf.keras.layers.Input(shape=item_input_size, dtype="float32")

    user_encoding = tf.keras.layers.CategoryEncoding(num_tokens=46000, output_mode="one_hot")(user_input)
    item_encoding = tf.keras.layers.CategoryEncoding(num_tokens=2565241, output_mode="one_hot")(item_input)

    user_embedding = tf.keras.layers.Dense(units=embedding_size, dtype="float32", 
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(user_encoding)
    item_embedding = tf.keras.layers.Dense(units=embedding_size, dtype="float32", 
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(item_encoding)

    x = tf.keras.layers.Concatenate()([user_embedding, item_embedding])
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32",
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [16]:
def get_gmf(num_users, num_items, latent_dim):
    
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    user_flatten = tf.keras.layers.Flatten()(user_embedding)
    item_flatten = tf.keras.layers.Flatten()(item_embedding)

    x = tf.keras.layers.Lambda(lambda x: tf.multiply(x[0], x[1]))([user_flatten, item_flatten])
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [17]:
def get_mlp(num_users, num_items, latent_dim, layer_dims):
    
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    user_flatten = tf.keras.layers.Flatten()(user_embedding)
    item_flatten = tf.keras.layers.Flatten()(item_embedding)

    x = tf.keras.layers.Concatenate()([user_flatten, item_flatten])

    # multi layers
    for i, l in enumerate(layer_dims):
        x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32", name=f'{i}st_layer',
            kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    # last layer
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [ ]:
def get_neuMF(num_users, num_items, latent_dim, layer_dims):

    # common inputs
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    # gmf branch
    gmf_user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='gmf_user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    gmf_item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='gmf_item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    gmf_user_flatten = tf.keras.layers.Flatten()(gmf_user_embedding)
    gmf_item_flatten = tf.keras.layers.Flatten()(gmf_item_embedding)

    gmf_multiply = tf.keras.layers.Lambda(lambda x: tf.multiply(x[0], x[1]), name='gmf_multiply')(
        [gmf_user_flatten, gmf_item_flatten])

    # mlp branch
    mlp_user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='mlp_user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    mlp_item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='mlp_item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    mlp_user_flatten = tf.keras.layers.Flatten()(mlp_user_embedding)
    mlp_item_flatten = tf.keras.layers.Flatten()(mlp_item_embedding)

    x = tf.keras.layers.Concatenate()([mlp_user_flatten, mlp_item_flatten])

    # multi layers
    for i, l in enumerate(layer_dims):
        x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32", name=f'{i}st_layer',
            kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)
        
    # concat branches
    x = tf.keras.layers.Concatenate()([gmf_multiply, x])

    # last layer
    neumf_last_layer = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42), bias_constraint=None)(x)

    neumf_model = tf.keras.Model([user_input, item_input], neumf_last_layer)


In [18]:
# def get_mlp(user_input_size, item_input_size, embedding_size, layer_units, predictive_factor_size):
    
#     user_input = tf.keras.layers.Input(shape=user_input_size, dtype="float32")
#     item_input = tf.keras.layers.Input(shape=item_input_size, dtype="float32")

#     user_encoding = tf.keras.layers.CategoryEncoding(num_tokens=46000, output_mode="one_hot")(user_input)
#     item_encoding = tf.keras.layers.CategoryEncoding(num_tokens=2565241, output_mode="one_hot")(item_input)

#     user_embedding = tf.keras.layers.Dense(units=embedding_size, dtype="float32", 
#         kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(user_encoding)
#     item_embedding = tf.keras.layers.Dense(units=embedding_size, dtype="float32", 
#         kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(item_encoding)

#     x = tf.keras.layers.Concatenate()([user_embedding, item_embedding])

#     for l in layer_units:
#         x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32",
#             kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)
#     x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32",
#         kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

#     model = tf.keras.Model([user_input, item_input], x)
#     return model

In [19]:
gmf_model = get_gmf(len(all_users), len(all_items), 16)
gmf_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 user_embedding (Embedding)     (None, 1, 16)        736000      ['input_1[0][0]']                
                                                                                                  
 item_embedding (Embedding)     (None, 1, 16)        41043856    ['input_2[0][0]']                
                                                                                              

2023-07-18 14:36:30.090652: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-18 14:36:30.090925: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-18 14:36:30.091260: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/usr/local/cuda-11.7/lib64:/usr/local/cuda/extras/CUPTI/:/usr/local/cuda-11.7/lib64:/usr/local/cuda/extras/CUPTI/
2023-07-18 14:36:30.091343: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dle

In [20]:
mlp_model = get_mlp(len(all_users), len(all_items), 16, [32, 16, 8])
mlp_model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 input_4 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 user_embedding (Embedding)     (None, 1, 16)        736000      ['input_3[0][0]']                
                                                                                                  
 item_embedding (Embedding)     (None, 1, 16)        41043856    ['input_4[0][0]']                
                                                                                            

In [21]:
# mlp_model = get_mlp(100, 5043, 16, [32, 16], 8)
# mlp_model.summary()

In [23]:
learning_rate = 0.00001
gmf_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy", metrics=['acc'])

In [22]:
learning_rate = 0.00001
mlp_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy", metrics=['acc'])

In [45]:
# gmf_model.fit([train_X[:, 0], train_X[:, 1]], train_y)

In [24]:
np.expand_dims(np.array(train_y, dtype='float32'), axis=1).shape

(5038484, 1)

In [25]:
train_X_user.shape

(5038484,)

# GMF (generalization matrix factorization)

In [26]:
gmf_model.fit([train_X_user, train_X_item], np.array(train_y, dtype='float32'), 
    epochs=1, batch_size=1024)

10077/10077 [==============================] - 1434s 142ms/step - loss: 0.6931 - acc: 0.4999


# Multi Layer Perceptron

In [27]:
mlp_model.fit([train_X_user, train_X_item], np.array(train_y, dtype='float32'), 
    epochs=1, batch_size=1024)

4921/4921 [==============================] - 689s 140ms/step - loss: 0.6931 - acc: 0.4995


# NeuMF (Neural Matrix Factorization)

In [24]:
train_X_user.shape, train_X_item.shape, len(train_y)

((5038484,), (5038484,), 5038484)

In [29]:
test_pred = gmf_model.predict([test_X_user, test_X_item])

2875/2875 [==============================] - 15s 5ms/step


In [27]:
from sklearn.metrics import accuracy_score

In [38]:
test_pred_2 = test_pred >= 0.5

In [39]:
accuracy_score(test_y, test_pred_2)

0.9989130434782608

In [40]:
from sklearn.metrics import ndcg_score

In [42]:
user_scores = {u:[] for u in all_users}
for u in zip(test_X_user, test_y):
    user_scores[u].append()

array([ 1005.,  2527., 43809., ..., 43994., 23269., 28389.])

In [ ]:
ndcg_score

---

In [ ]:
@tf.function
def hit_ratio(y_true, y_pred):
    y_pred = [int(x >= 0.5) for x in y_pred]
    hits = [int(x==y==1) for x,y in zip(y_pred, y_true)]
    return sum(hits)/sum(y_pred)

In [ ]:
hit_ratio([1, 0, 1], [0.2, 0.1, 0.9])

<tf.Tensor: shape=(), dtype=float32, numpy=1.0>

In [ ]:
import tensorflow as tf
tf.reduce_sum([0,0,1])

<tf.Tensor: shape=(), dtype=int32, numpy=1>

In [ ]:
# class hit_ratio(tf.keras.metrics.Metric):
#     def __init__(self, name = 'hit_ratio', **kwargs):
#         super(hit_ratio, self).__init__(**kwargs)
#         self.hits = None

#     def update_state(self, y_true, y_pred, sample_weight=None):
#         y_true = tf.cast(y_true, tf.bool)
#         y_pred = tf.math.greater(y_pred, 0.5)
#         self.hits = tf.logical_and(tf.equal(y_true, y_pred), tf.equal(y_true, 1))
#         self.targets = tf.reduce_sum(y_pred)
        
#     def reset_state(self):
#         self.hits.assign(0)

#     def result(self):
#         return tf.reduced_sum(self.hits)/sum(y_pred)

In [ ]:
# def hr_keras(y_true, y_pred):
#     score = tf.py_function(func=hit_ratio, inp=[y_true, y_pred], Tout=tf.float64,  name='custom_hr') # tf 2.x
#     #score = tf.py_func( lambda y_true, y_pred : mse_AIFrenz(y_true, y_pred) , [y_true, y_pred], 'float32', stateful = False, name = 'custom_mse' ) # tf 1.x
#     return score